# AI应用基础 · 工单M4.2《工业智能决策助手系统》 — 教师参考版
第11周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m4kb"], check=True)
print("数据已生成: data/")

工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础
M4.2 -> data/knowledge_base.json 13 条知识
完成。数据输出目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础/data
数据已生成: data/


## 参考实现 `m42`（含依赖导入）

In [2]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m42():
    import re
    import time
    import requests

    BASE = "http://127.0.0.1:8000"

    def chat(msgs):
        last = None
        for i in range(3):                          # 重试: 覆盖服务冷启动窗口
            try:
                r = requests.post(f"{BASE}/v1/chat/completions",
                                  json={"messages": msgs}, timeout=5)
                r.raise_for_status()
                return r.json()["choices"][0]["message"]["content"]
            except requests.RequestException as e:
                last = e
                time.sleep(1 + i)
        raise last

    kb = json.load(open("data/knowledge_base.json", encoding="utf-8"))
    docs = [t for v in kb.values() for t in v]

    def retrieve(q, topk=2):
        qs = set(re.findall(r"[\u4e00-\u9fa5a-zA-Z0-9]+", q))
        return sorted(docs, key=lambda t: -len(qs & set(
            re.findall(r"[\u4e00-\u9fa5a-zA-Z0-9]+", t))))[:topk]

    qs = ["泵振动7.5mm/s怎么处理？", "轴承温度82℃要不要停机？",
          "变频器过流报警如何排查？"]
    for q in qs:
        ctx = "\n".join(retrieve(q))
        a = chat([{"role": "system", "content": "CONTEXT:" + ctx},
                  {"role": "user", "content": q}])
        print("Q:", q, "\nA:", a[:90], "\n")
    print("Agent演示: 振动7.5>4.5阈值 → 判定超标 → 检索知识库 → 生成停机检查建议。")


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `4` | TODO-4: RAG对比: 检索结果注入后再问, 记录是否更"有依据" TODO-5: |
|  | Agent全流程问答记录表(问题/回答/依据) |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 运行环境(如需)

In [3]:
# 启动本地模拟LLM服务(Litestar+granian, 后台子进程)
import subprocess, sys, time, atexit, urllib.request
p = subprocess.Popen([sys.executable, "数据与工具/mock_llm_server.py", "8000"])
atexit.register(lambda: p.terminate())
for _ in range(30):                       # 就绪探针: /health 返回ok才继续
    try:
        if b"ok" in urllib.request.urlopen("http://127.0.0.1:8000/health",
                                           timeout=1).read():
            print("mock LLM on :8000"); break
    except Exception:
        time.sleep(0.5)
else:
    print("警告: mock LLM 未就绪, 后续调用将失败")

[INFO] Starting granian (main PID: 82035)
[INFO] Listening at: http://0.0.0.0:8000
[INFO] Spawning worker-1 with PID: 82037


[INFO] Started worker-1
mock LLM on :8000


## 执行与验证

In [4]:
results = m42()

Q: 泵振动7.5mm/s怎么处理？ 
A: 根据知识库检索到以下依据：
泵类设备振动速度有效值超过4.5mm/s应列入重点关注，超过7.1mm/s必须停机检查。
轴承温度报警阈值设定为75℃，达到85℃应立即停机并检查润滑与 

Q: 轴承温度82℃要不要停机？ 
A: 根据知识库检索到以下依据：
泵类设备振动速度有效值超过4.5mm/s应列入重点关注，超过7.1mm/s必须停机检查。
轴承温度报警阈值设定为75℃，达到85℃应立即停机并检查润滑与 

Q: 变频器过流报警如何排查？ 
A: 根据知识库检索到以下依据：
泵类设备振动速度有效值超过4.5mm/s应列入重点关注，超过7.1mm/s必须停机检查。
轴承温度报警阈值设定为75℃，达到85℃应立即停机并检查润滑与 

Agent演示: 振动7.5>4.5阈值 → 判定超标 → 检索知识库 → 生成停机检查建议。
